# Notebook 01: Ingesta y limpieza de datos

**Proyecto:** Análisis y estimación del riesgo de robos y hurtos en CABA  
**Objetivo:** Ingerir el dataset de delitos de 2023, auditar su calidad, descartar registros no aptos para el análisis espacial, normalizar las coordenadas y exportar una versión limpia y trazable.

**Criterio geográfico acordado:**
- Se descartan coordenadas vacías o no numéricas.
- Se descartan coordenadas `0,0`, utilizadas como marcador de ausencia.
- Se descartan registros con coordenadas reales pero sin barrio/comuna; los 31 casos fueron revisados en el mapa y están fuera de CABA.

## 1. Configuración del entorno

Se importan las librerías necesarias y se definen rutas compatibles con la estructura del repositorio:

```text
datasets/
├── raw/
└── processed/
notebooks/
```

El notebook también incluye una ruta alternativa a `/mnt/data` para poder probarlo en ChatGPT o Colab sin modificar el código principal.

In [1]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# Detección de la raíz del proyecto
# ------------------------------------------------------------
# Si el notebook se ejecuta desde la carpeta "notebooks", la raíz
# del proyecto es su carpeta padre. Si se ejecuta desde la raíz,
# se utiliza directamente el directorio actual.
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Estructura principal del proyecto
RAW_DATA_DIR = PROJECT_ROOT / "datasets" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "datasets" / "processed"

# Se crea la carpeta de salida si todavía no existe.
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Localización del archivo original
# ------------------------------------------------------------
# El primer nombre es el recomendado dentro del repositorio.
# Los demás son alternativas para copias descargadas o pruebas.
raw_file_candidates = [
    RAW_DATA_DIR / "delitos_2023.csv",
    PROJECT_ROOT / "data" / "raw" / "delitos_2023.csv",
    Path("/mnt/raw/delitos_2023.csv"),
]

RAW_FILE = next(
    (path for path in raw_file_candidates if path.exists()),
    None
)

if RAW_FILE is None:
    expected_paths = "\n".join(f" - {path}" for path in raw_file_candidates)
    raise FileNotFoundError(
        "No se encontró el archivo original de delitos. "
        "Ubicalo en alguna de estas rutas:\n"
        f"{expected_paths}"
    )

# Archivos de salida
CLEAN_FILE = PROCESSED_DATA_DIR / "delitos_2023_caba_limpio.csv"
DISCARDED_FILE = PROCESSED_DATA_DIR / "delitos_2023_descartados.csv"
SUMMARY_FILE = PROCESSED_DATA_DIR / "resumen_limpieza_delitos_2023.csv"

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Archivo de entrada: {RAW_FILE}")
print(f"Carpeta de salida: {PROCESSED_DATA_DIR}")

Raíz del proyecto: C:\Users\nikko\CienciaDatosDelitosTPO
Archivo de entrada: C:\Users\nikko\CienciaDatosDelitosTPO\datasets\raw\delitos_2023.csv
Carpeta de salida: C:\Users\nikko\CienciaDatosDelitosTPO\datasets\processed


## 2. Ingesta del dataset original

Se carga el archivo fuente sin aplicar filtros. En esta etapa se conserva la totalidad de los registros para poder auditar después cada descarte.

In [2]:
raw_df = pd.read_csv(
    RAW_FILE,
    low_memory=False
)

print(f"Filas cargadas: {len(raw_df):,}")
print(f"Columnas cargadas: {raw_df.shape[1]}")
display(raw_df.head())

Filas cargadas: 155,897
Columnas cargadas: 15


,id-mapa,anio,mes,dia,fecha,franja,tipo,subtipo,uso_arma,uso_moto,barrio,comuna,latitud,longitud,cantidad
0,1114512,2023,ENERO,DOMINGO,2023-01-01,1,Amenazas,Amenazas,NO,NO,NaN,NaN,0.0,0.0,1
1,1114513,2023,ENERO,LUNES,2023-01-02,12,Amenazas,Amenazas,NO,NO,NaN,NaN,0.0,0.0,1
2,1114514,2023,ENERO,MARTES,2023-01-03,7,Amenazas,Amenazas,NO,NO,NaN,NaN,0.0,0.0,1
3,1114515,2023,ENERO,MARTES,2023-01-03,21,Amenazas,Amenazas,NO,NO,NaN,NaN,0.0,0.0,1
4,1114516,2023,ENERO,MARTES,2023-01-03,12,Amenazas,Amenazas,NO,NO,NaN,NaN,0.0,0.0,1


## 3. Diagnóstico inicial y perfilado de datos

Se controlan:

- columnas obligatorias;
- identificadores duplicados;
- coordenadas vacías o no numéricas;
- coordenadas `0,0`;
- registros con coordenadas pero sin barrio/comuna;
- distribución inicial por tipo de delito.

En este notebook, **nulo técnico** significa una celda vacía o convertida a `NaN` [Not a Number, valor faltante].  
El valor `0,0` no es nulo técnicamente, pero se considera **geolocalización no utilizable**.

In [3]:
# Columnas imprescindibles para la limpieza y el análisis posterior.
required_columns = {
    "id-mapa",
    "anio",
    "mes",
    "dia",
    "fecha",
    "franja",
    "tipo",
    "subtipo",
    "uso_arma",
    "uso_moto",
    "barrio",
    "comuna",
    "latitud",
    "longitud",
    "cantidad",
}

missing_columns = sorted(required_columns - set(raw_df.columns))

if missing_columns:
    raise ValueError(
        f"Faltan columnas obligatorias en el dataset: {missing_columns}"
    )

# Conversión auxiliar: todavía no modifica el dataset original.
lat_numeric = pd.to_numeric(raw_df["latitud"], errors="coerce")
lon_numeric = pd.to_numeric(raw_df["longitud"], errors="coerce")

barrio_aux = raw_df["barrio"].astype("string").str.strip()
comuna_aux = raw_df["comuna"].astype("string").str.strip()

mask_empty_coordinates = lat_numeric.isna() | lon_numeric.isna()

mask_zero_coordinates = (
    ~mask_empty_coordinates
    & (
        lat_numeric.eq(0)
        | lon_numeric.eq(0)
    )
)

mask_missing_territory = (
    barrio_aux.isna()
    | barrio_aux.eq("")
    | comuna_aux.isna()
    | comuna_aux.eq("")
)

mask_coordinates_without_territory = (
    ~mask_empty_coordinates
    & ~mask_zero_coordinates
    & mask_missing_territory
)

diagnostic_summary = pd.DataFrame(
    {
        "control": [
            "Registros totales",
            "ID duplicados",
            "Coordenadas vacías o no numéricas",
            "Coordenadas 0,0",
            "Coordenadas reales sin barrio/comuna",
            "Registros inicialmente aptos para análisis espacial",
        ],
        "cantidad": [
            len(raw_df),
            raw_df["id-mapa"].duplicated().sum(),
            mask_empty_coordinates.sum(),
            mask_zero_coordinates.sum(),
            mask_coordinates_without_territory.sum(),
            (
                ~mask_empty_coordinates
                & ~mask_zero_coordinates
                & ~mask_missing_territory
            ).sum(),
        ],
    }
)

display(diagnostic_summary)

type_distribution = (
    raw_df["tipo"]
    .value_counts(dropna=False)
    .rename_axis("tipo")
    .reset_index(name="cantidad")
)

type_distribution["porcentaje"] = (
    type_distribution["cantidad"] / len(raw_df) * 100
).round(2)

display(type_distribution)

,control,cantidad
0,Registros totales,155897
1,ID duplicados,0
2,Coordenadas vacías o no numéricas,348
3,"Coordenadas 0,0",2779
4,Coordenadas reales sin barrio/comuna,31
5,Registros inicialmente aptos para análisis esp...,152739


,tipo,cantidad,porcentaje
0,Robo,64983,41.68
1,Hurto,62567,40.13
2,Vialidad,10430,6.69
3,Lesiones,9612,6.17
4,Amenazas,8214,5.27
5,Homicidios,91,0.06


## 4. Limpieza y estructuración

La limpieza sigue un criterio conservador y auditable:

1. Se normalizan nombres y tipos de columnas.
2. Se identifican por separado los motivos de descarte.
3. Se preserva una copia de las coordenadas originales.
4. Se normalizan las coordenadas, que en el archivo fuente tienen el separador decimal implícito.
5. Se agregan variables temporales simples para el análisis posterior.
6. Los registros excluidos se guardan en un archivo separado; no desaparecen sin explicación.

No se imputan coordenadas. Inventar una ubicación para un delito sería estadísticamente cómodo y metodológicamente bastante indefendible.

In [4]:
# Copia de trabajo: el dataframe original queda intacto.
df = raw_df.copy()

# ------------------------------------------------------------
# 4.1. Estandarización de nombres y tipos
# ------------------------------------------------------------
df = df.rename(columns={"id-mapa": "id_mapa"})

text_columns = [
    "mes",
    "dia",
    "tipo",
    "subtipo",
    "uso_arma",
    "uso_moto",
    "barrio",
]

for column in text_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

# Fechas y columnas numéricas
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["anio"] = pd.to_numeric(df["anio"], errors="coerce").astype("Int64")
df["franja"] = pd.to_numeric(df["franja"], errors="coerce").astype("Int64")
df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").astype("Int64")
df["comuna"] = pd.to_numeric(df["comuna"], errors="coerce").astype("Int64")

# Se conservan los valores originales para auditoría.
df["latitud_original"] = df["latitud"]
df["longitud_original"] = df["longitud"]

df["latitud"] = pd.to_numeric(df["latitud"], errors="coerce")
df["longitud"] = pd.to_numeric(df["longitud"], errors="coerce")

# Un error de fecha u hora no se imputa automáticamente.
invalid_dates = df["fecha"].isna().sum()
invalid_hours = (~df["franja"].between(0, 23)).sum()

if invalid_dates:
    raise ValueError(f"Se encontraron {invalid_dates} fechas inválidas.")

if invalid_hours:
    raise ValueError(f"Se encontraron {invalid_hours} horas fuera del rango 0-23.")

# ------------------------------------------------------------
# 4.2. Máscaras de descarte
# ------------------------------------------------------------
mask_empty_coordinates = (
    df["latitud"].isna()
    | df["longitud"].isna()
)

mask_zero_coordinates = (
    ~mask_empty_coordinates
    & (
        df["latitud"].eq(0)
        | df["longitud"].eq(0)
    )
)

mask_missing_territory = (
    df["barrio"].isna()
    | df["comuna"].isna()
)

# Estos 31 casos fueron revisados visualmente y están fuera de CABA.
mask_outside_caba = (
    ~mask_empty_coordinates
    & ~mask_zero_coordinates
    & mask_missing_territory
)

# Control adicional de duplicados por identificador.
# En la versión analizada del dataset no existen duplicados,
# pero el control queda incorporado para futuras actualizaciones.
mask_duplicate_id = df["id_mapa"].duplicated(keep="first")

# ------------------------------------------------------------
# 4.3. Motivo de descarte
# ------------------------------------------------------------
df["motivo_descarte"] = pd.Series(pd.NA, index=df.index, dtype="string")

df.loc[
    mask_empty_coordinates,
    "motivo_descarte"
] = "Coordenadas vacías o no numéricas"

df.loc[
    mask_zero_coordinates & df["motivo_descarte"].isna(),
    "motivo_descarte"
] = "Coordenadas 0,0"

df.loc[
    mask_outside_caba & df["motivo_descarte"].isna(),
    "motivo_descarte"
] = "Coordenadas fuera de CABA / sin barrio y comuna"

df.loc[
    mask_duplicate_id & df["motivo_descarte"].isna(),
    "motivo_descarte"
] = "ID duplicado"

discard_mask = df["motivo_descarte"].notna()

discarded_df = df.loc[discard_mask].copy()
clean_df = df.loc[~discard_mask].copy()

# ------------------------------------------------------------
# 4.4. Normalización de coordenadas
# ------------------------------------------------------------
def normalize_implicit_decimal(value, maximum_absolute_value):
    '''
    Restaura el separador decimal implícito de las coordenadas.

    Ejemplo:
        -34625121  -> -34.625121
        -58348646  -> -58.348646

    Se divide por una potencia de diez hasta que el valor entra
    dentro del rango matemáticamente posible de latitud o longitud.
    '''
    if pd.isna(value) or value == 0:
        return np.nan

    value = float(value)

    if abs(value) <= maximum_absolute_value:
        return value

    power = math.ceil(
        math.log10(abs(value) / maximum_absolute_value)
    )

    return value / (10 ** power)


clean_df["latitud"] = clean_df["latitud"].map(
    lambda value: normalize_implicit_decimal(value, 90)
)

clean_df["longitud"] = clean_df["longitud"].map(
    lambda value: normalize_implicit_decimal(value, 180)
)

# ------------------------------------------------------------
# 4.5. Variables temporales derivadas
# ------------------------------------------------------------
clean_df["franja_horaria"] = pd.cut(
    clean_df["franja"],
    bins=[-1, 5, 11, 17, 23],
    labels=["MADRUGADA", "MAÑANA", "TARDE", "NOCHE"],
    ordered=True,
)

clean_df["es_fin_de_semana"] = (
    clean_df["dia"]
    .isin(["SABADO", "DOMINGO"])
    .astype("int8")
)

# En el dataset limpio no hace falta conservar el motivo de descarte.
clean_df = clean_df.drop(columns=["motivo_descarte"])

print(f"Registros limpios: {len(clean_df):,}")
print(f"Registros descartados: {len(discarded_df):,}")
display(
    discarded_df["motivo_descarte"]
    .value_counts()
    .rename_axis("motivo")
    .reset_index(name="cantidad")
)

Registros limpios: 152,739
Registros descartados: 3,158


,motivo,cantidad
0,"Coordenadas 0,0",2779
1,Coordenadas vacías o no numéricas,348
2,Coordenadas fuera de CABA / sin barrio y comuna,31


## 5. Validación y exportación

Antes de guardar los archivos se verifican las cantidades esperadas para la versión original de 2023:

- 348 registros con coordenadas vacías;
- 2.779 registros con coordenadas `0,0`;
- 31 registros geolocalizados fuera de CABA;
- 3.158 descartes en total;
- 152.739 registros válidos.

Las validaciones estrictas solo se aplican cuando el archivo contiene exactamente 155.897 filas. Así, una actualización futura del dataset no falla por tener cantidades diferentes, pero sí emite una advertencia.

In [5]:
# ------------------------------------------------------------
# 5.1. Controles de integridad
# ------------------------------------------------------------
assert clean_df["latitud"].notna().all()
assert clean_df["longitud"].notna().all()
assert clean_df["latitud"].ne(0).all()
assert clean_df["longitud"].ne(0).all()
assert clean_df["barrio"].notna().all()
assert clean_df["comuna"].notna().all()
assert clean_df["id_mapa"].is_unique

# Rango amplio de Buenos Aires: detecta una normalización rota
# sin confundirlo con una prueba geométrica del límite de CABA.
assert clean_df["latitud"].between(-35, -34).all()
assert clean_df["longitud"].between(-59, -58).all()

# Controles exactos para la versión analizada.
if len(raw_df) == 155_897:
    expected_counts = {
        "Coordenadas vacías o no numéricas": 348,
        "Coordenadas 0,0": 2_779,
        "Coordenadas fuera de CABA / sin barrio y comuna": 31,
    }

    actual_counts = (
        discarded_df["motivo_descarte"]
        .value_counts()
        .to_dict()
    )

    for reason, expected in expected_counts.items():
        actual = actual_counts.get(reason, 0)
        assert actual == expected, (
            f"Cantidad inesperada para '{reason}': "
            f"se esperaban {expected:,} y se obtuvieron {actual:,}."
        )

    assert len(discarded_df) == 3_158
    assert len(clean_df) == 152_739
else:
    warnings.warn(
        "El archivo no tiene 155.897 filas. "
        "Se ejecutaron controles generales, pero no las cantidades "
        "exactas de la versión analizada."
    )

# ------------------------------------------------------------
# 5.2. Resumen de la limpieza
# ------------------------------------------------------------
cleaning_summary = pd.DataFrame(
    {
        "categoria": [
            "Registros originales",
            "Coordenadas vacías o no numéricas",
            "Coordenadas 0,0",
            "Coordenadas fuera de CABA / sin barrio y comuna",
            "ID duplicados",
            "Total descartado",
            "Registros válidos",
            "Porcentaje conservado",
        ],
        "valor": [
            len(raw_df),
            mask_empty_coordinates.sum(),
            mask_zero_coordinates.sum(),
            mask_outside_caba.sum(),
            mask_duplicate_id.sum(),
            len(discarded_df),
            len(clean_df),
            round(len(clean_df) / len(raw_df) * 100, 2),
        ],
    }
)

display(cleaning_summary)

# ------------------------------------------------------------
# 5.3. Exportación
# ------------------------------------------------------------
# UTF-8 con BOM facilita la apertura correcta en Excel.
clean_df.to_csv(
    CLEAN_FILE,
    index=False,
    encoding="utf-8-sig"
)

discarded_df.to_csv(
    DISCARDED_FILE,
    index=False,
    encoding="utf-8-sig"
)

cleaning_summary.to_csv(
    SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Archivos generados correctamente:")
print(f" - Dataset limpio: {CLEAN_FILE}")
print(f" - Registros descartados: {DISCARDED_FILE}")
print(f" - Resumen de limpieza: {SUMMARY_FILE}")

,categoria,valor
0,Registros originales,155897.00
1,Coordenadas vacías o no numéricas,348.00
2,"Coordenadas 0,0",2779.00
3,Coordenadas fuera de CABA / sin barrio y comuna,31.00
4,ID duplicados,0.00
5,Total descartado,3158.00
6,Registros válidos,152739.00
7,Porcentaje conservado,97.97


Archivos generados correctamente:
 - Dataset limpio: C:\Users\nikko\CienciaDatosDelitosTPO\datasets\processed\delitos_2023_caba_limpio.csv
 - Registros descartados: C:\Users\nikko\CienciaDatosDelitosTPO\datasets\processed\delitos_2023_descartados.csv
 - Resumen de limpieza: C:\Users\nikko\CienciaDatosDelitosTPO\datasets\processed\resumen_limpieza_delitos_2023.csv
